# 01 - Data cleaning
**Goal:** turn the messy raw files into trustworthy tables, and log every change.

Prerequisite: run `python src/01_generate_data.py`. Cleaning rules live in `src/cleaning.py`; this notebook shows *why* each is needed.

*Rule of thumb:* (1) fix keys and types, (2) remove duplicates, (3) fix impossible values, (4) fill missing values - in that order.

In [1]:
# --- setup: make the project's src/ folder importable and set paths ---
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display, Markdown
import config as cfg
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
N = json.loads((cfg.TABLES / "key_numbers.json").read_text())   # every number the scripts computed
%matplotlib inline

In [2]:
raw = {p.stem: pd.read_csv(p) for p in sorted(cfg.DATA_RAW.glob("*.csv"))}
for name, df in raw.items():
    print(f"{name:<18} rows={len(df):>8,}  columns={df.shape[1]}  null cells={int(df.isna().sum().sum()):,}")

demand_daily       rows= 131,695  columns=6  null cells=394
inventory_daily    rows= 131,433  columns=5  null cells=395
inventory_policy   rows=     180  columns=7  null cells=0
products           rows=      92  columns=7  null cells=3
purchase_orders    rows=   7,012  columns=12  null cells=219
suppliers          rows=      24  columns=6  null cells=2
warehouses         rows=       5  columns=4  null cells=0


## 1. Look at the problems in the raw data
We never trust a file before profiling it. Each line checks one kind of problem.

In [3]:
d = raw["demand_daily"]
print("Duplicate sku-warehouse-day rows         :", int(d.duplicated(["sku", "warehouse_code", "date"]).sum()))
print("Rows where sku is not in 'SKU-00012' form:", int((~d["sku"].astype(str).str.fullmatch(r"SKU-\d{5}")).sum()))
print("Dates NOT in ISO format                  :", int((~d["date"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}")).sum()))
print("Negative units_demanded                  :", int((d["units_demanded"] < 0).sum()))
print("Missing units_demanded                   :", int(d["units_demanded"].isna().sum()))
print("units_fulfilled > units_demanded         :", int((d["units_fulfilled"] > d["units_demanded"]).sum()))
display(d.sample(6, random_state=1))

Duplicate sku-warehouse-day rows         : 655
Rows where sku is not in 'SKU-00012' form: 1316
Dates NOT in ISO format                  : 1314
Negative units_demanded                  : 131
Missing units_demanded                   : 394
units_fulfilled > units_demanded         : 392


,sku,warehouse_code,date,units_demanded,units_fulfilled,promo_flag
36837,SKU-00053,WH-APAC,2024-06-23,52.0,52,0
62031,SKU-00035,WH-APAC,2025-11-22,0.0,0,0
64288,SKU-00074,WH-EAST,2025-02-22,2.0,2,0
104799,SKU-00084,WH-EU,2024-07-03,0.0,0,0
11866,SKU-00015,WH-WEST,2024-02-23,4.0,4,0
69881,SKU-00011,WH-EU,2024-02-10,2.0,2,0


In [4]:
print("Category spellings in raw products:")
display(raw["products"]["category"].value_counts())

Category spellings in raw products:


category
Raw Materials       22
Chemicals           17
Electronics         15
Textiles             9
Packaging            9
Machinery Parts      7
CHEMICALS            3
MACHINERY PARTS      3
TEXTILES             3
packaging            1
textiles             1
electronics          1
chemicals            1
Name: count, dtype: int64

## 2. Apply the cleaning rules
`cleaning.clean_all` runs one small function per table and returns a log of every rule.

In [5]:
import cleaning
clean, log = cleaning.clean_all(raw)
display(log[log["rows_affected"] > 0].sort_values("rows_affected", ascending=False))

,table,issue,rule,rows_affected,action
11,demand_daily,SKU code has lowercase/extra spaces,strip + upper-case,1316,standardised
13,demand_daily,Dates in a second format (dd-Mon-yyyy),explicit format parsing,1314,converted to ISO dates
12,demand_daily,Warehouse code has lowercase/extra spaces,strip + upper-case,657,standardised
15,demand_daily,Duplicate sku-warehouse-day rows,drop_duplicates on the business key,655,dropped
22,inventory_daily,Duplicate sku-warehouse-day rows,drop_duplicates on the business key,393,dropped
17,demand_daily,Missing units_demanded,fill with units_fulfilled (lower bound of true...,393,imputed
24,inventory_daily,Missing on_hand_units,carry forward last known stock level (stock is...,393,imputed
19,demand_daily,units_fulfilled > units_demanded (impossible),cap fulfilled at demanded,279,corrected
1,purchase_orders,ship_mode in mixed case,lower-case,139,standardised ('AIR' -> 'air')
23,inventory_daily,Negative stock on hand,set to 0 (physical stock cannot be negative),131,corrected


## 3. Prove the result
After cleaning, the business key must be unique, values valid, and categories consistent.

In [6]:
assert not clean["demand_daily"].duplicated(["sku", "warehouse_code", "date"]).any()
assert (clean["demand_daily"]["units_demanded"] >= 0).all()
assert (clean["demand_daily"]["units_fulfilled"] <= clean["demand_daily"]["units_demanded"]).all()
assert (clean["inventory_daily"]["on_hand_units"] >= 0).all()
assert clean["products"]["sku"].is_unique and clean["products"]["unit_cost"].notna().all()
print("Categories after cleaning:", sorted(clean["products"]["category"].unique()))
display(cleaning.data_quality_summary(raw, clean))

Categories after cleaning: ['Chemicals', 'Electronics', 'Machinery Parts', 'Packaging', 'Raw Materials', 'Textiles']


,table,rows_raw,rows_clean,null_cells_raw,null_cells_clean
0,warehouses,5,5,0,0
1,purchase_orders,7012,6991,219,680
2,products,92,90,3,0
3,suppliers,24,24,2,0
4,demand_daily,131695,131040,394,0
5,inventory_daily,131433,131040,395,0
6,inventory_policy,180,180,0,0


In [7]:
fixed = int(log.loc[~log["issue"].str.contains("check only|integrity", case=False), "rows_affected"].sum())
worst = log.sort_values("rows_affected", ascending=False).iloc[0]
dups = int(log.loc[log["issue"].str.contains("Duplicate sku-warehouse-day") & (log["table"] == "demand_daily"), "rows_affected"].sum())
display(Markdown(f"""### So what
* **{fixed:,} records** were corrected or removed; the most common issue was *{worst['issue']}* ({int(worst['rows_affected']):,} rows).
* Duplicates alone removed {dups:,} demand rows - left in, they would double count demand and distort fill rate.
* **Action:** add validation at the source (unique keys, non-negative quantities, ISO dates, controlled category list). The full log is in `outputs/tables/cleaning_log.csv`."""))

### So what
* **5,998 records** were corrected or removed; the most common issue was *SKU code has lowercase/extra spaces* (1,316 rows).
* Duplicates alone removed 655 demand rows - left in, they would double count demand and distort fill rate.
* **Action:** add validation at the source (unique keys, non-negative quantities, ISO dates, controlled category list). The full log is in `outputs/tables/cleaning_log.csv`.